# EDA — exploration du jeu de données brut (BRONZE)

**Livrable J1.** Première exploration avant toute décision de nettoyage ou de modélisation.

Ce notebook est **exploratoire et jetable**. Tout ce qui en sort et qui doit survivre a été
industrialisé dans `src/` : le nettoyage dans `pipeline/cleaning.py`, les contrôles dans
`quality_checks.py`, les variables du modèle dans `ml/features.py`.

> Un notebook qui reste la source de vérité d'un traitement est une dette, pas un livrable.


In [ ]:
import sys, warnings
from pathlib import Path
from datetime import UTC, datetime

sys.path.insert(0, str(Path.cwd().parent / 'src'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.width', 120)


## 1. Génération du lot Bronze

Graine fixe et ancrage temporel : deux exécutions produisent les mêmes octets.
C'est ce qui rend cette exploration reproductible par un tiers.


In [ ]:
from banking_dataops.generate_synthetic_data import SyntheticConfig, generate

BRONZE = Path.cwd().parent / 'data'
generate(SyntheticConfig(
    customers=100, accounts_per_customer=2, transactions=1500, seed=42,
    generated_at=datetime(2026, 8, 3, 6, 45, tzinfo=UTC),
), BRONZE)

tx = pd.read_csv(BRONZE / 'synthetic_transactions.csv')
cu = pd.read_csv(BRONZE / 'synthetic_customers.csv')
ac = pd.read_csv(BRONZE / 'synthetic_accounts.csv')

print(f'transactions {tx.shape}  clients {cu.shape}  comptes {ac.shape}')


## 2. Volumétrie, types, valeurs manquantes


In [ ]:
tx.info()
print()
print('Valeurs manquantes par colonne :')
print(tx.isna().sum()[lambda s: s > 0] if tx.isna().any().any() else '  aucune')


In [ ]:
tx.describe(include='all').T


## 3. Distribution des montants

Attendu : une log-normale. C'est ce qui motive la variable `amount_log` dans le modèle —
un modèle linéaire lit très mal une distribution aussi asymétrique.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(tx['amount_chf'], bins=60, ax=axes[0])
axes[0].set_title('Montants — échelle linéaire')
sns.histplot(np.log1p(tx['amount_chf']), bins=60, ax=axes[1], color='#1B9E5A')
axes[1].set_title('Montants — log1p')
plt.tight_layout(); plt.show()

print(tx['amount_chf'].describe())
print(f"\nAsymétrie : {tx['amount_chf'].skew():.2f}  (0 = symétrique)")


## 4. Variables catégorielles


In [ ]:
for col in ['channel', 'merchant_category', 'country', 'status', 'source_system']:
    counts = tx[col].value_counts()
    print(f'{col} — {len(counts)} modalités')
    print((counts / len(tx) * 100).round(1).to_string())
    print()


## 5. La cible : `is_suspicious`

**Point décisif de l'exploration.** La cible est déséquilibrée. C'est ce qui condamne
l'exactitude comme métrique : prédire « jamais suspect » scorerait déjà plus de 90 %.


In [ ]:
rate = tx['is_suspicious'].mean()
print(f'Taux de positifs : {rate:.1%}')
print(f"Exactitude d'un modèle qui prédit toujours 'non' : {1 - rate:.1%}")
print('\n-> métriques retenues : précision, rappel, F1, ROC AUC.')


### 5.1 Fuite de cible — le point qui a changé la conception

`is_suspicious` est dérivé d'un seuil sur `risk_score`. Donner `risk_score` au modèle
produirait une performance parfaite sans aucun apprentissage.

La vérification ci-dessous est la raison pour laquelle `risk_score` figure dans
`FORBIDDEN_FEATURES`, protégé par une assertion qui lève.


In [ ]:
seuil = tx.loc[tx['is_suspicious'], 'risk_score'].min()
print(f'Score minimal parmi les suspects  : {seuil:.4f}')
print(f'Score maximal parmi les non-suspects : {tx.loc[~tx["is_suspicious"], "risk_score"].max():.4f}')
print('\n-> séparation parfaite : risk_score EST la cible. Variable interdite.')


## 6. Facteurs de risque observables

Ce que le modèle peut légitimement apprendre, une fois `risk_score` écarté.


In [ ]:
tx['hour'] = pd.to_datetime(tx['event_timestamp']).dt.hour
tx['is_night'] = tx['hour'] < 6

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

(tx.groupby('is_night')['is_suspicious'].mean() * 100).plot(
    kind='bar', ax=axes[0], color=['#0F1D3D', '#D64545'])
axes[0].set_title('Taux de suspicion — nuit vs jour (%)'); axes[0].set_xlabel('')

(tx.groupby('channel')['is_suspicious'].mean() * 100).sort_values().plot(
    kind='barh', ax=axes[1], color='#1C3461')
axes[1].set_title('Taux de suspicion par canal (%)')

(tx.groupby('merchant_category')['is_suspicious'].mean() * 100).sort_values().plot(
    kind='barh', ax=axes[2], color='#1C3461')
axes[2].set_title('Taux de suspicion par catégorie (%)')

plt.tight_layout(); plt.show()


## 7. Corrélations

Sur les seules variables numériques. La corrélation entre `risk_score` et `is_suspicious`
doit être écrasante — c'est la fuite, vue autrement.


In [ ]:
num = tx[['amount_chf', 'risk_score', 'hour', 'is_night', 'is_suspicious']].astype(float)
corr = num.corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Matrice de corrélation'); plt.tight_layout(); plt.show()

print(f"corr(risk_score, is_suspicious) = {corr.loc['risk_score','is_suspicious']:.3f}  <- la fuite")


## 8. Qualité — ce que l'exploration révèle et qui devient un contrôle

Chaque constat ci-dessous a été transformé en contrôle industrialisé.


In [ ]:
print('Doublons d\'identifiant   :', tx['transaction_id'].duplicated().sum(), ' -> CTRL-002')
print('Montants hors bornes      :', ((tx['amount_chf'] <= 0) | (tx['amount_chf'] > 1e6)).sum(), ' -> CTRL-003')
print('Scores hors [0,1]         :', ((tx['risk_score'] < 0) | (tx['risk_score'] > 1)).sum(), ' -> CTRL-005')
print('Statuts hors référentiel  :', (~tx['status'].isin(['posted','pending','rejected'])).sum(), ' -> CTRL-006')
print('Comptes orphelins         :', (~tx['account_id'].isin(ac['account_id'])).sum(), ' -> CTRL-004')


## 9. Conclusions de l'exploration

| Constat | Conséquence sur la conception |
|---|---|
| Montants log-normaux | Variable `amount_log` ajoutée |
| Cible à ~7 % de positifs | Exactitude écartée, split stratifié, `class_weight='balanced'` |
| `risk_score` sépare parfaitement la cible | Variable **interdite**, assertion dans `features.py` |
| Nuit, canal et catégorie discriminent | Retenus comme variables observables |
| Aucun doublon, aucun orphelin en Bronze | Le schéma les empêche déjà — CTRL-001/002/004 sont structurels |

**Suite :** `python run_pipeline.py --train`. Rien de ce notebook n'est exécuté en production.
